# ANOVAFSelector

Scores each feature alone with the ANOVA F statistic, then keeps the highest.

F compares how far apart the class means are with how spread out each class
is. A large F means the feature's average differs clearly between classes.

- **Cost:** one pass over the data. Fast even with thousands of features.
- **Strength:** simple, well understood, gives a p-value per feature.
- **Weakness:** it only sees differences in the *mean*. A feature that
  separates classes in any other way scores near zero.
- **Modes:** `n_features=k` keeps the top k; `n_features=None` keeps every
  feature with p-value below `alpha`.

In [1]:
import numpy as np

rng = np.random.default_rng(0)
from sklearn.datasets import make_classification

from featsel.selectors import ANOVAFSelector

# 500 features, the first 10 informative (shuffle=False keeps them first).
X, y = make_classification(n_samples=300, n_features=500, n_informative=10,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(10))

## Top k

In [2]:
selector = ANOVAFSelector(n_features=10).fit(X, y)
chosen = selector.get_support(indices=True)
print('chosen            :', sorted(chosen.tolist()))
print('truly informative :', len(truth & set(chosen)), 'of 10')

chosen            : [0, 4, 5, 7, 8, 30, 76, 273, 321, 417]
truly informative : 5 of 10


## Letting the p-value decide the count

In [3]:
selector = ANOVAFSelector(n_features=None, alpha=0.001).fit(X, y)
chosen = selector.get_support(indices=True)
print(f'{len(chosen)} features pass p < 0.001; {len(truth & set(chosen))} are truly informative')

5 features pass p < 0.001; 4 are truly informative


## Where it fails

Here the class depends on how *far* a feature is from zero, not on its sign.
Both classes have the same mean, so F is near zero and the feature is missed.

In [4]:
x = rng.normal(size=1000)
y_far = (np.abs(x) > 1).astype(int)
noise = rng.normal(size=(1000, 49))
X_far = np.column_stack([x, noise])

rank = np.argsort(ANOVAFSelector(n_features=5).fit(X_far, y_far).scores_)[::-1]
print('rank of the one informative feature:', int(np.where(rank == 0)[0][0]) + 1, 'of 50')

rank of the one informative feature: 17 of 50
